# Regression Testing for a Conversational Agent

Prompt changes cause regressions, and `assert output == expected` doesn't work on free text.
Instead: a suite of **simulated conversations**, scored automatically.

**System under test:** a phone routing agent for the fictional property manager
"Stadtquartier Wohnservice". A tenant calls, the agent asks questions and routes to a department.

| Role | Function | Job |
| --- | --- | --- |
| System under test | `route`, `ask_question` | the agent we protect |
| Test agent | `tenant_reply` | plays the tenant from a persona |
| Judge | `judge_friendliness` | scores the agent's tone 1–5 |

A scenario **passes** only if all three hold:
1. `routing_correct` — right department (deterministic)
2. `within_budget` — tenant turns ≤ `max_turns` (deterministic)
3. `friendliness ≥ min_friendliness` — LLM judge

**Usage:** run §1–§6 for a baseline → §7 breaks the prompt → re-run §6 → §8 repairs it → re-run §6.

## 1. Setup and Test Scenarios

`DEPARTMENTS` is the single source of truth for the routing taxonomy.
Each scenario is one test case. `expected_department` has two special values:
`"auth-failed"` (call must be stopped) and `None` (any department is fine).

| Case | Probes |
| --- | --- |
| TC-001 | happy path |
| TC-002 | second department, calm caller |
| TC-003 | frustrated caller — agent must stay warm |
| TC-004 | unverifiable address — call must end |
| TC-005 | vague input — agent must ask, not guess |
| TC-006 | semantic trap: "cancel" but it's energy |

In [ ]:
import os
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"        # agent + test agent
JUDGE_MODEL = "gpt-5-mini"  # pinned separately so scores stay comparable if MODEL changes

DEPARTMENTS = {
    "rental-contracts":     "Rental Contracts — lease questions, renewals, amendments",
    "terminations-moveout": "Terminations & Move-out — cancellations, move-out dates, deposit returns",
    "tenant-complaints":    "Tenant Complaints — noise, neighbour disputes, general grievances",
    "energy-heating":       "Energy & Heating — heating failures, hot water, utility billing",
    "repairs-maintenance":  "Repairs & Maintenance — broken fixtures, structural damage, general repairs",
}

TEST_SCENARIOS = [
    {
        "id": "TC-001",
        "description": "Clear heating issue, cooperative tenant",
        "persona": "You are Lisa Müller, Berliner Str. 12, Berlin. Your heating stopped working 2 days ago. "
                   "Polite but worried. Give name and address when asked. Keep answers short.",
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 3,
    },
    {
        "id": "TC-002",
        "description": "Lease termination, calm and direct",
        "persona": "You are Thomas Bauer, Hauptstr. 7, Hamburg. You want to end your lease. "
                   "Calm and businesslike. Give name and address when asked.",
        "expected_department": "terminations-moveout",
        "max_turns": 5,
        "min_friendliness": 2,
    },
    {
        "id": "TC-003",
        "description": "Noise complaint, frustrated tone",
        "persona": "You are Emre Yilmaz, Gartenweg 3, Munich. Your upstairs neighbour makes noise every night "
                   "and you haven't slept in a week. Frustrated and impatient. Give details when asked.",
        "expected_department": "tenant-complaints",
        "max_turns": 6,
        "min_friendliness": 3,
    },
    {
        "id": "TC-004",
        "description": "Auth failure — wrong address",
        "persona": "You are Sandra Koch. You want to report a broken window. When asked for your address, "
                   "say 'Musterstr. 99, Berlin'. Do not correct it, even if it can't be verified.",
        "expected_department": "auth-failed",
        "max_turns": 3,
        "min_friendliness": 3,
    },
    {
        "id": "TC-005",
        "description": "Ambiguous issue — agent must ask, not guess",
        "persona": "You are Klaus Werner, Friedrichstr. 45, Berlin. Your first message is only: "
                   "'I have a problem with my apartment.' Give more details only when asked directly.",
        "expected_department": None,
        "max_turns": 6,
        "min_friendliness": 3,
    },
    {
        "id": "TC-006",
        "description": "Tricky phrasing — sounds like terminations but is energy",
        "persona": "You are Yuki Tanaka, Rosenweg 8, Frankfurt. You want to cancel your energy supply "
                   "contract, not your lease. Say: 'I want to cancel my energy contract.' Give details when asked.",
        "expected_department": "energy-heating",
        "max_turns": 5,
        "min_friendliness": 2,
    },
]

## 2. The Routing Agent (System Under Test)

* **Structured output:** `RoutingDecision` is a Pydantic schema, so the test asserts on
  `.department` instead of parsing prose. `Literal` types restrict it to valid values.
* **`confidence`** lets the agent keep asking instead of guessing (TC-005).
* `ROUTING_SYSTEM_PROMPT` is what the suite protects; `route()` reads it at call time,
  so §7/§8 can swap it.

In [ ]:
class RoutingDecision(BaseModel):
    department: Literal[tuple(DEPARTMENTS)]
    routing_reason: str
    issue_summary: str
    confidence: Literal["low", "medium", "high"]


GOOD_ROUTING_PROMPT = (
    "You are a routing agent at Stadtquartier Wohnservice, a residential property management company.\n"
    "Based on the conversation transcript, decide which department should handle this tenant's issue.\n\n"
    "Available departments:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DEPARTMENTS.items())
    + "\n\nChoose exactly one department key from the list above. "
    "In routing_reason, explain specifically why this department is the right fit."
)
ROUTING_SYSTEM_PROMPT = GOOD_ROUTING_PROMPT


def format_transcript(transcript: list[dict]) -> str:
    return "\n".join(f"{t['role'].upper()}: {t['text']}" for t in transcript)


def route(transcript: list[dict]) -> RoutingDecision | None:
    """Routing decision for the conversation so far, or None on API error."""
    try:
        response = client.chat.completions.parse(
            model=MODEL,
            messages=[
                {"role": "system", "content": ROUTING_SYSTEM_PROMPT},
                {"role": "user", "content": f"Conversation:\n{format_transcript(transcript)}"},
            ],
            response_format=RoutingDecision,
        )
        return response.choices[0].message.parsed
    except Exception as e:
        print(f"  routing error: {e}")
        return None


def ask_question(transcript: list[dict]) -> str:
    """One short clarifying question to collect name, address or details of the issue."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a friendly receptionist at Stadtquartier Wohnservice. "
                "Ask for the tenant's name, address, or clarify their issue. One short question."
            )},
            {"role": "user", "content": format_transcript(transcript)},
        ],
    )
    return response.choices[0].message.content.strip()

## 3. Test Agent (Simulated Tenant)

A second LLM with the persona as its system prompt. The roles are flipped: from its point
of view, the support agent is the `user` and its own past lines are `assistant`.

In [ ]:
def tenant_reply(persona: str, transcript: list[dict]) -> str:
    messages = [{"role": "system", "content": (
        f"{persona}\n\nYou are on a customer support call. Respond naturally as this person, "
        "1–3 sentences. Do not break character."
    )}]
    for t in transcript:
        messages.append({"role": "assistant" if t["role"] == "tenant" else "user", "content": t["text"]})

    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content.strip()

## 4. Conversation Runner

Per turn: tenant speaks → auth check → (from turn 2) routing attempt → if no confident
routing, the agent asks one more question. Capped at `max_turns + 2` turns so it always ends.

The auth check is a deterministic stub (unknown-address list) so TC-004 stays stable and free.

In [ ]:
GREETING = "Thank you for calling Stadtquartier Wohnservice. My name is Chappy, how can I help you today?"
AUTH_FAILED_REPLY = (
    "I'm really sorry — I couldn't find that address in our tenant records, so I can't take your request "
    "by phone. Please check the address on your lease and call us again, or email us — we'll be glad to help."
)
UNKNOWN_ADDRESSES = ["musterstr"]  # stand-in for a tenant-database lookup


def address_unknown(transcript: list[dict]) -> bool:
    tenant_text = " ".join(t["text"] for t in transcript if t["role"] == "tenant").lower()
    return any(a in tenant_text for a in UNKNOWN_ADDRESSES)


def run_conversation(scenario: dict) -> dict:
    transcript = [{"role": "agent", "text": GREETING}]
    routing, auth_failed = None, False

    for _ in range(scenario["max_turns"] + 2):
        transcript.append({"role": "tenant", "text": tenant_reply(scenario["persona"], transcript)})

        if address_unknown(transcript):
            auth_failed = True
            transcript.append({"role": "agent", "text": AUTH_FAILED_REPLY})
            break

        if len(transcript) >= 5:  # greeting + 2 exchanges
            decision = route(transcript)
            # Only hand off when the agent is sure enough. decision is None on an API error;
            # a "low" decision is discarded and the agent asks another question below.
            # Only the final hand-off is evaluated: never confident -> "no-routing" -> FAIL.
            if decision and decision.confidence != "low":
                routing = decision
                team = DEPARTMENTS[decision.department].split(" — ")[0]
                transcript.append({"role": "agent", "text": f"Thank you. I'm routing you to our {team} team."})
                break

        transcript.append({"role": "agent", "text": ask_question(transcript)})

    return {"transcript": transcript, "routing": routing, "auth_failed": auth_failed}

## 5. Evaluator

Two deterministic checks, one LLM judge — use the judge only for what code can't check.
The rubric anchors 1/3/5 so scores don't drift between runs.

In [ ]:
class Judgement(BaseModel):
    score: Literal[1, 2, 3, 4, 5]
    reason: str


FRIENDLINESS_RUBRIC = """Score the AGENT's tone on a scale of 1 to 5:
  1 = cold or dismissive
  3 = professional but flat
  5 = warm and empathetic
Judge tone only — not helpfulness, correctness or company policy."""


def judge_friendliness(transcript: list[dict]) -> Judgement:
    response = client.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": FRIENDLINESS_RUBRIC},
            {"role": "user", "content": format_transcript(transcript)},
        ],
        response_format=Judgement,
    )
    return response.choices[0].message.parsed


def evaluate(scenario: dict, result: dict) -> dict:
    routing, auth_failed = result["routing"], result["auth_failed"]
    expected = scenario["expected_department"]
    actual = routing.department if routing else ("auth-failed" if auth_failed else "no-routing")

    if expected is None:
        routing_correct = routing is not None
    else:
        routing_correct = actual == expected

    turns = sum(t["role"] == "tenant" for t in result["transcript"])
    within_budget = turns <= scenario["max_turns"]
    judgement = judge_friendliness(result["transcript"])

    return {
        "id": scenario["id"],
        "expected": expected or "any",
        "actual": actual,
        "turns": turns,
        "friendliness": judgement.score,
        "routing_correct": routing_correct,
        "within_budget": within_budget,
        "passed": routing_correct and within_budget and judgement.score >= scenario["min_friendliness"],
        "judge_reason": judgement.reason,
    }

## 6. Run the Suite

Track the summary metrics after every prompt change. Expect ±1 scenario of jitter
between runs; a trend across runs is a real regression.

In [ ]:
def run_suite(scenarios: list[dict]) -> pd.DataFrame:
    rows = []
    for s in scenarios:
        row = evaluate(s, run_conversation(s))
        rows.append(row)
        print(f"{'✅' if row['passed'] else '❌'} {s['id']} {s['description']} — "
              f"{row['actual']} | turns {row['turns']} | friendliness {row['friendliness']}/5")

    df = pd.DataFrame(rows)
    print(f"\nPASSED {df.passed.sum()}/{len(df)} | routing accuracy {df.routing_correct.mean():.0%} | "
          f"avg turns {df.turns.mean():.1f} | avg friendliness {df.friendliness.mean():.1f}/5")
    return df


results = run_suite(TEST_SCENARIOS)
results

## 7. Break Something

A well-meant "speed-up": someone adds a keyword shortcut for cancellations.
Re-run §6 — TC-006 ("cancel my **energy** contract") should now fail, while friendliness stays flat.
A tone metric alone would have stayed green.

In [ ]:
ROUTING_SYSTEM_PROMPT = GOOD_ROUTING_PROMPT + (
    "\n\nShortcut: if the tenant wants to cancel, terminate or end any contract, "
    "always route to terminations-moveout."
)

## 8. Fix It

Restore the original prompt and re-run §6. Green → red → green proves the suite actually detects regressions.

In [ ]:
ROUTING_SYSTEM_PROMPT = GOOD_ROUTING_PROMPT